# Home Credit PD Scorecard Validation and Business Use

This notebook takes the scorecard outputs from the build notebook and turns them into a **portfolio-ready validation and business interpretation pack**.

## What this notebook does
- evaluates discrimination using **AUC, Gini and KS**
- reviews score distributions and risk bands
- builds **decile / rank-order tables**
- checks **calibration**
- proposes a simple **approval and review strategy**
- translates model outputs into business language for interviews and GitHub

This is the notebook that shows the scorecard is not just a model, but a usable credit decision tool.

## 1. Load saved scorecard artefacts

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

PROCESSED_DIR = Path("../processed")
OUTPUT_DIR = PROCESSED_DIR / "scorecard_outputs"

TARGET = "TARGET"
ID_COL = "SK_ID_CURR"

iv_summary = pd.read_csv(OUTPUT_DIR / "01_iv_summary.csv")
woe_table = pd.read_csv(OUTPUT_DIR / "02_woe_table.csv")
coef_table = pd.read_csv(OUTPUT_DIR / "03_scorecard_coefficients.csv")
scorecard_points = pd.read_csv(OUTPUT_DIR / "04_scorecard_points.csv")
train_scored = pd.read_csv(OUTPUT_DIR / "05_train_scored.csv")
test_scored = pd.read_csv(OUTPUT_DIR / "06_test_scored.csv")
scorecard_metadata = pd.read_csv(OUTPUT_DIR / "07_scorecard_metadata.csv")

scorecard_metadata

## 2. Core validation metrics

AUC is useful, but it is not enough on its own.  
A bank-style validation summary usually also looks at **Gini** and **KS**.

In [ ]:
def ks_statistic(y_true, pd_pred):
    df_ks = pd.DataFrame({"y_true": y_true, "pd_pred": pd_pred}).sort_values("pd_pred", ascending=False).reset_index(drop=True)
    df_ks["good"] = 1 - df_ks["y_true"]
    df_ks["bad"] = df_ks["y_true"]
    df_ks["cum_good"] = df_ks["good"].cumsum() / df_ks["good"].sum()
    df_ks["cum_bad"] = df_ks["bad"].cumsum() / df_ks["bad"].sum()
    df_ks["ks_gap"] = np.abs(df_ks["cum_bad"] - df_ks["cum_good"])
    return df_ks["ks_gap"].max(), df_ks

test_auc = roc_auc_score(test_scored[TARGET], test_scored["pd_pred"])
test_gini = 2 * test_auc - 1
test_ks, ks_table = ks_statistic(test_scored[TARGET], test_scored["pd_pred"])

validation_summary = pd.DataFrame({
    "metric": ["AUC", "Gini", "KS"],
    "value": [test_auc, test_gini, test_ks]
})

validation_summary

In [ ]:
plt.figure(figsize=(7, 6))
fpr, tpr, _ = roc_curve(test_scored[TARGET], test_scored["pd_pred"])
plt.plot(fpr, tpr, linewidth=2, label=f"ROC (AUC = {test_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5, label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Scorecard Validation")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
plt.plot(ks_table["cum_bad"], label="Cumulative bads", linewidth=2)
plt.plot(ks_table["cum_good"], label="Cumulative goods", linewidth=2)
plt.title(f"KS Curve (KS = {test_ks:.3f})")
plt.xlabel("Sorted observations by predicted PD")
plt.ylabel("Cumulative distribution")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Rank ordering by score decile

A good scorecard should show clear rank ordering:
- safer score buckets should have lower bad rates
- riskier score buckets should have higher bad rates

That is one of the easiest ways to explain scorecard quality to non-technical stakeholders.

In [ ]:
test_scored["score_decile"] = pd.qcut(test_scored["score"], q=10, duplicates="drop")
decile_table = (
    test_scored.groupby("score_decile", observed=False)
    .agg(
        observations=(TARGET, "size"),
        defaults=(TARGET, "sum"),
        bad_rate=(TARGET, "mean"),
        avg_score=("score", "mean"),
        avg_pd=("pd_pred", "mean"),
    )
    .reset_index()
)

decile_table = decile_table.sort_values("avg_score", ascending=False).reset_index(drop=True)
decile_table

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(decile_table.index + 1, decile_table["bad_rate"], marker="o", linewidth=2)
plt.xticks(decile_table.index + 1)
plt.xlabel("Score decile (1 = safest average score)")
plt.ylabel("Observed bad rate")
plt.title("Observed Bad Rate by Score Decile")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Score-band performance

This table turns the scorecard into something closer to a policy tool.

The goal is not to pretend that score bands are final production cut-offs.  
The goal is to show how model outputs can support:
- approval
- manual review
- decline / senior credit review

In [ ]:
band_summary = (
    test_scored.groupby("score_band", observed=False)
    .agg(
        observations=(TARGET, "size"),
        defaults=(TARGET, "sum"),
        bad_rate=(TARGET, "mean"),
        avg_score=("score", "mean"),
        avg_pd=("pd_pred", "mean"),
    )
    .reset_index()
    .sort_values("score_band", ascending=False)
)

band_summary

## 5. Calibration check

Discrimination tells us whether the scorecard ranks risk well.  
Calibration tells us whether predicted PD is reasonably aligned with realised default rates.

For a portfolio notebook, a simple grouped calibration view is enough.

In [ ]:
calibration_table = (
    test_scored.assign(pd_bucket=pd.qcut(test_scored["pd_pred"], q=10, duplicates="drop"))
    .groupby("pd_bucket", observed=False)
    .agg(
        observations=(TARGET, "size"),
        avg_predicted_pd=("pd_pred", "mean"),
        observed_bad_rate=(TARGET, "mean"),
    )
    .reset_index()
)

calibration_table

In [ ]:
plt.figure(figsize=(7, 6))
plt.plot(calibration_table["avg_predicted_pd"], calibration_table["observed_bad_rate"], marker="o", linewidth=2)
max_axis = max(calibration_table["avg_predicted_pd"].max(), calibration_table["observed_bad_rate"].max())
plt.plot([0, max_axis], [0, max_axis], linestyle="--", linewidth=1.5, label="Perfect calibration")
plt.xlabel("Average predicted PD")
plt.ylabel("Observed bad rate")
plt.title("Grouped Calibration Plot")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Variable interpretation

This is the section that helps in interviews.

Instead of just showing coefficients, explain the business story:
- which variables drive risk up
- which variables reduce risk
- why that makes sense in credit terms

In [ ]:
top_positive = coef_table.sort_values("coefficient", ascending=False).head(8).copy()
top_negative = coef_table.sort_values("coefficient", ascending=True).head(8).copy()

print("Top positive risk drivers:")
display(top_positive)

print("\nTop negative risk drivers:")
display(top_negative)

### Suggested business interpretation

Use this kind of language in GitHub or interviews:

- **Repayment burden variables** increase risk because borrowers with stretched cash flow have less room to absorb shocks.
- **Leverage variables** increase risk because higher debt relative to collateral or income usually weakens resilience.
- **Behavioural arrears variables** increase risk because recent payment friction often appears before formal default.
- **External credit score variables** usually reduce risk because they capture broader repayment quality outside the application file.
- **Stability variables** such as age or employment length often indicate a more established borrower profile.

The exact final interpretation should be confirmed using your fitted coefficient table and WOE trend review.

## 7. Simple policy strategy overlay

A model becomes more business-relevant once you show how it could be used in a decision process.

This section creates a simple example:
- **A / B bands** → fast-track approval or standard approval
- **C band** → manual review
- **D / E bands** → decline or senior review

These are portfolio illustration rules, not final production policy.

In [ ]:
policy_table = band_summary.copy()

policy_table["suggested_action"] = np.select(
    [
        policy_table["score_band"].isin(["A", "B"]),
        policy_table["score_band"].isin(["C"]),
        policy_table["score_band"].isin(["D", "E"]),
    ],
    [
        "Approve / standard credit path",
        "Manual review / analyst judgement",
        "Decline or escalate to senior credit",
    ],
    default="Review"
)

policy_table

## 8. Executive summary for GitHub README or interviews

This notebook supports the following story:

> I first built an enhanced exploratory PD model on the Home Credit dataset.  
> Then I translated the predictive model into a governed scorecard workflow using WOE binning, IV review and logistic regression on WOE variables.  
> Finally, I validated the scorecard using AUC, Gini, KS, decile analysis and grouped calibration, and showed how score bands can support approval and review decisions.

That is a much stronger portfolio story than stopping at raw logistic regression.

## 9. Save validation outputs

In [ ]:
validation_summary.to_csv(OUTPUT_DIR / "08_validation_summary.csv", index=False)
decile_table.to_csv(OUTPUT_DIR / "09_decile_table.csv", index=False)
band_summary.to_csv(OUTPUT_DIR / "10_score_band_summary.csv", index=False)
calibration_table.to_csv(OUTPUT_DIR / "11_calibration_table.csv", index=False)
policy_table.to_csv(OUTPUT_DIR / "12_policy_table.csv", index=False)

print("Saved validation outputs to:", OUTPUT_DIR.resolve())
sorted([p.name for p in OUTPUT_DIR.iterdir()])

## 10. Next improvement ideas

Once this scorecard is working end to end, the most sensible next upgrades are:
1. monotonic manual bin review
2. population stability index by score band
3. reject inference discussion
4. time-based validation split
5. calibration to long-run or through-the-cycle PD
6. score-to-grade mapping with portfolio default benchmarks

Those are stronger next steps than simply chasing a slightly higher AUC.